#### Delta Lake Format in PySpark/Databricks

Delta Lake is an open-source storage layer that brings ACID transactions and reliability to data lakes.

##### Key Features:
1. ACID Transactions
2. Time Travel (Versioning)



###### 1. ACID Transactions
- Atomicity: Operations either fully succeed or fully fail
- Consistency: Data remains valid across concurrent operations
- Isolation: Concurrent reads/writes don't interfere
- Durability: Committed changes persist

##### 2. Time Travel (Versioning)
### 

In [0]:
# Query historical versions
spark.read.format("delta").option("versionAsOf", 0).load("/path/to/delta")
spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/path")

# SQL syntax
spark.sql("SELECT * FROM retail.orders VERSION AS OF 5")
spark.sql("SELECT * FROM retail.orders TIMESTAMP AS OF '2024-01-01'")

###### 3. Schema Evolution

In [0]:
# Merge schema on write
df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/path/to/delta")

###### 4. UPSERT/MERGE Operations

In [0]:
MERGE INTO retail.orders AS target
USING updates AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

###### 5. Optimized Performance
- Z-Ordering: Co-locate related data for faster queries
- Data Skipping: Automatic min/max statistics
- Compaction: OPTIMIZE command consolidates small files

In [0]:
%sql
-- Optimize table
OPTIMIZE retail.orders;

-- Z-order by frequently filtered columns
OPTIMIZE retail.orders ZORDER BY (customer_id, order_date);

###### 6. Deletes & Updates

Unlike Parquet/CSV, Delta supports true DML:

In [0]:
%sql
DELETE FROM retail.orders WHERE status = 'cancelled';
UPDATE retail.orders SET status = 'completed' WHERE order_id = 5;

###### Delta vs Parquet
![image_1775715073049.png](./image_1775715073049.png "image_1775715073049.png")

###### Delta File Structure
![image_1775715199863.png](./image_1775715199863.png "image_1775715199863.png") 

###### Common Delta Operations
Vacuum (Cleanup old files)

In [0]:
%sql
-- Remove files older than 7 days (default retention)
VACUUM retail.orders;

-- Custom retention
VACUUM retail.orders RETAIN 168 HOURS;

History & Details

In [0]:
%sql
-- View transaction history
DESCRIBE HISTORY retail.orders;

-- View table details (size, files, partitions)
DESCRIBE DETAIL retail.orders;

Restore to Previous Version


In [0]:
%sql
RESTORE TABLE retail.orders TO VERSION AS OF 3;
RESTORE TABLE retail.orders TO TIMESTAMP AS OF '2024-01-01';

- Default format in Databricks: Delta is the default table format
- Managed tables: Use USING delta (as in your code)
- External tables: Delta works with LOCATION clause
- Streaming: Delta is ideal for streaming ETL pipelines
- INSERT OVERWRITE: With Delta, it's an atomic operation (old data visible until new data is committed)
- Transaction log: _delta_log/ directory stores all operations
- No small files problem: OPTIMIZE consolidates files automatically

###### Creating Delta Tables

In [0]:
# Method 1: SQL DDL
spark.sql("""
    CREATE TABLE retail.products (
        product_id INT,
        name STRING,
        price DOUBLE
    ) USING delta
    PARTITIONED BY (category STRING)
""")

# Method 2: DataFrame API
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail.products")

# Method 3: Path-based
df.write.format("delta") \
    .mode("append") \
    .save("/dbfs/delta/products")

###### Why Delta in Production?
- Data integrity: ACID guarantees prevent corrupted data
- Auditability: Full transaction history for compliance
- Performance: 10-100x faster queries with optimization
- Simplified ETL: Native streaming + batch support
- Cost savings: Fewer files = lower storage/metadata costs
- Delta Lake transforms your data lake into a lakehouse architecture, combining the best of data lakes (scalability, low cost) and data warehouses (reliability, performance).

###### Default Table Format Without USING delta
1. Databricks Environment
Default: Delta Lake

2. Open-Source Spark
Default: Parquet

In [0]:
# These are equivalent in Databricks
spark.sql("CREATE TABLE products (id INT, name STRING)")
# Same as:
spark.sql("CREATE TABLE products (id INT, name STRING) USING delta")

In [0]:
# Without USING clause
spark.sql("CREATE TABLE products (id INT, name STRING)")
# Same as:
spark.sql("CREATE TABLE products (id INT, name STRING) USING parquet")

###### Configuration Setting
 The default is controlled by:

In [0]:
# This works universally across all Spark versions
all_configs = spark.conf.getAll

# Display
for key, value in all_configs.items():
    print(f"{key} = {value}")

##### ACID guarantees in Delta Lake
Delta Lake achieves ACID transactions through its transaction log (the _delta_log directory). Every write to a Delta table — insert, update, delete, merge — creates a new JSON entry in the log. This log is the single source of truth for what the table looks like at any point in time.

![image_1777213345308.png](./image_1777213345308.png "image_1777213345308.png")

> Every commit is atomic — it either fully succeeds or fully fails. There are no partial writes.

##### The two isolation levels
**Snapshot Isolation — the default**
Every transaction sees a consistent snapshot of the table taken at the moment the transaction begins. Concurrent writers don't block each other — they each work against their own snapshot and race to commit.


In [0]:
# Job A and Job B start at the same time
# Both see version 5 of the table

# Job A reads and writes — commits as version 6
# Job B reads and writes — tries to commit as version 6
# → conflict detected → Job B retries against version 6

If two jobs modify different parts of the table (different partitions, different rows), both commits succeed. If they conflict on the same data, Delta Lake raises a ConcurrentModificationException and one job must retry.

**Serializable — stricter, optional**

Serializable isolation guarantees that the outcome of concurrent transactions is identical to some sequential execution of those same transactions — as if they ran one after another with no overlap at all.

In [0]:
# Enable Serializable isolation on a table
spark.sql("""
  ALTER TABLE my_table
  SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')
""")

Serializable catches a class of anomaly that Snapshot Isolation can miss — specifically write skew, where two transactions each read data and make a decision based on what they saw, but the combination of their writes produces an inconsistent result.


##### The key difference — write skew
Here's a concrete example:

![image_1777213645316.png](./image_1777213645316.png "image_1777213645316.png")


Both jobs read a valid state and made a locally correct decision, but together they violated the invariant. Snapshot Isolation doesn't catch this. Serializable does.


##### How DeltaLake detects conflicts?

![image_1777212856055.png](./image_1777212856055.png "image_1777212856055.png")

##### Optimistic concurrency — how Delta resolves conflicts
Delta Lake uses optimistic concurrency control — transactions don't acquire locks during reads. They only check for conflicts at commit time. This is what makes Delta Lake efficient for concurrent writes.
The conflict resolution logic at commit time follows three steps:

1. Read the current version of the transaction log
2. Check if any committed transaction since your snapshot - touched the same files you're about to modify
3. No overlap → commit succeeds → new version written
4. Overlap detected → raise ConcurrentModificationException → caller must retry the transaction

![image_1777213998534.png](./image_1777213998534.png "image_1777213998534.png")